# LAB1 — Self-RAG com Tokens de Controle via vLLM

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 8 — Self-RAG e CRAG: Recuperação Reflexiva e Adaptativa**

---

## Objetivo

Neste laboratório, vamos explorar o conceito de **Self-RAG** (Self-Reflective Retrieval-Augmented Generation), que introduz **tokens de controle especiais** na geração de texto para que o modelo decida *autonomamente* quando buscar documentos externos, quando o contexto recuperado é relevante e quando a resposta gerada é útil e fundamentada.

O backend de inferência utilizado é o **vLLM** — um servidor de alta performance que expõe uma API compatível com OpenAI, permitindo rodar LLMs open-source localmente no Colab com aceleração em GPU.

### Tokens de Controle do Self-RAG

| Token | Significado | Ação |
|-------|-------------|------|
| `[Retrieve]` | O modelo avalia se precisa buscar documentos | Ativa ou não o retrieval |
| `[ISREL]` | Is Relevant — o documento recuperado é relevante? | Filtra documentos irrelevantes |
| `[ISSUP]` | Is Supported — a resposta é suportada pelos documentos? | Verifica fundamentação |
| `[ISUSE]` | Is Useful — a resposta é útil para a pergunta? | Avalia utilidade final |

### Fluxo do Self-RAG

```
Pergunta → [Retrieve=YES/NO] → Se YES: Busca docs → [ISREL] → Gera resposta → [ISSUP] → [ISUSE]
                             → Se NO: Gera direto → [ISSUP] → [ISUSE]
```

### Por que vLLM?

O vLLM oferece:
- **PagedAttention**: gerenciamento de memória eficiente para múltiplas requisições simultâneas
- **API OpenAI-compatible**: integra com LangChain, LlamaIndex e qualquer cliente OpenAI sem mudanças
- **Suporte nativo ao modelo Self-RAG**: `selfrag/selfrag_llama2_7b` pode ser carregado diretamente
- **Throughput alto**: ideal para os múltiplos loops de avaliação do Self-RAG

### Aplicação no Contexto Jurídico

No domínio do Direito e Segurança Pública, o Self-RAG é especialmente valioso porque:
- Perguntas sobre **artigos específicos** exigem retrieval de legislação
- Perguntas sobre **conceitos gerais** podem ser respondidas sem busca
- O token `[ISSUP]` garante que a resposta está **fundamentada na lei**, não em "alucinações"

---
## PASSO 1 — Instalação das Dependências

Instalamos o vLLM e as bibliotecas do ecossistema LangChain:
- **vllm**: servidor de inferência com API OpenAI-compatible
- **langchain-openai**: cliente LangChain que consome APIs OpenAI (inclusive vLLM local)
- **chromadb**: banco vetorial local para indexação e busca
- **sentence-transformers**: modelo de embeddings multilíngue

In [ ]:
# Instala as bibliotecas Python necessárias
# O flag -q suprime a saída verbosa do pip
!pip install vllm langchain langchain-community langchain-openai chromadb sentence-transformers -q

print("✅ Bibliotecas Python instaladas com sucesso!")

In [ ]:
import subprocess
import time
import urllib.request

# ─── Configuração do servidor vLLM ────────────────────────────────────────────
# Opção A (recomendada para este lab): Llama 3.2 1B — leve, roda no Colab T4
# Opção B (Self-RAG genuíno): 'selfrag/selfrag_llama2_7b' — requer GPU A100
# Para usar o Self-RAG genuíno, troque MODELO abaixo e ajuste max-model-len.

MODELO = "meta-llama/Llama-3.2-1B-Instruct"   # Troque por selfrag/selfrag_llama2_7b para Self-RAG nativo
PORTA  = 8000

print(f"Iniciando servidor vLLM...")
print(f"  Modelo : {MODELO}")
print(f"  Porta  : {PORTA}")
print("  (O download do modelo pode levar alguns minutos na primeira execução)")

# Inicia o servidor vLLM como subprocesso em background
# --dtype auto        : usa float16 automaticamente na GPU Colab T4
# --max-model-len 4096: limita contexto para economizar VRAM
# --gpu-memory-utilization 0.90: usa 90% da VRAM disponível
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODELO,
        "--port", str(PORTA),
        "--dtype", "auto",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Aguarda o servidor ficar disponível (polling no endpoint /health)
print("\nAguardando servidor ficar disponível...")
for tentativa in range(36):  # Máximo 3 minutos (36 × 5s)
    try:
        urllib.request.urlopen(f"http://localhost:{PORTA}/health")
        print(f"\n✅ Servidor vLLM pronto! (porta {PORTA})")
        break
    except Exception:
        print(f"  [{(tentativa+1)*5}s] Aguardando...", end="\r")
        time.sleep(5)
else:
    print("\n⚠️ Timeout — verifique se há GPU disponível e tente reiniciar o servidor.")

---
## PASSO 2 — Preparar o Corpus Jurídico

Carregamos nosso corpus de documentos jurídicos. O código tenta primeiro carregar do arquivo JSON do dataset da aula, e se não encontrar, usa uma versão embutida (hardcoded) com os mesmos documentos.

In [ ]:
import json
import os

# Caminho para o arquivo de corpus do dataset da aula
CORPUS_PATH = "../datasets/corpus_juridico_aula8.json"

# Corpus hardcoded como fallback — garante que o lab funciona sem o arquivo externo
CORPUS_HARDCODED = [
    {"id": "doc001", "titulo": "CC - Art. 186 - Ato Ilícito",
     "texto": "Art. 186. Aquele que, por ação ou omissão voluntária, negligência ou imprudência, violar direito e causar dano a outrem, ainda que exclusivamente moral, comete ato ilícito.",
     "fonte": "Lei nº 10.406/2002 - Código Civil", "categoria": "responsabilidade_civil"},
    {"id": "doc002", "titulo": "CC - Art. 206 - Prazos Prescricionais",
     "texto": "Art. 206. Prescreve: §3º Em três anos: V - a pretensão de reparação civil.",
     "fonte": "Lei nº 10.406/2002 - Código Civil", "categoria": "prescricao"},
    {"id": "doc003", "titulo": "Súmula 227 STJ - Pessoa Jurídica e Dano Moral",
     "texto": "A pessoa jurídica pode sofrer dano moral.",
     "fonte": "STJ - Súmula nº 227", "categoria": "dano_moral"},
    {"id": "doc004", "titulo": "CP - Art. 121 - Homicídio Qualificado",
     "texto": "Art. 121. Matar alguém: Pena - reclusão, de 6 a 20 anos. §2º Qualificado: mediante paga, motivo torpe, meio cruel, traição ou para garantir outro crime. Pena: 12 a 30 anos.",
     "fonte": "Decreto-Lei nº 2.848/1940 - Código Penal", "categoria": "direito_penal"},
    {"id": "doc005", "titulo": "Lei Maria da Penha - Art. 5º",
     "texto": "Art. 5º Configura violência doméstica qualquer ação baseada no gênero que cause morte, lesão, sofrimento físico, sexual ou psicológico e dano moral ou patrimonial, no âmbito da unidade doméstica, da família ou em relação íntima de afeto.",
     "fonte": "Lei nº 11.340/2006", "categoria": "violencia_domestica"},
    {"id": "doc006", "titulo": "CF/88 - Art. 5º - Direitos Fundamentais",
     "texto": "Art. 5º Todos são iguais perante a lei, sendo invioláveis a intimidade, a vida privada, a honra e a imagem das pessoas, assegurado direito a indenização pelo dano moral decorrente de sua violação. O preso será informado de seus direitos, entre os quais o de permanecer calado.",
     "fonte": "Constituição Federal de 1988", "categoria": "direitos_fundamentais"},
    {"id": "doc007", "titulo": "Lei de Improbidade - Art. 9º",
     "texto": "Art. 9º Constitui improbidade por enriquecimento ilícito auferir qualquer vantagem patrimonial indevida em razão do exercício de cargo, mandato, função ou emprego público.",
     "fonte": "Lei nº 8.429/1992", "categoria": "improbidade_administrativa"},
    {"id": "doc008", "titulo": "CPP - Art. 302 - Flagrante Delito",
     "texto": "Art. 302. Considera-se em flagrante delito quem: I - está cometendo a infração penal; II - acaba de cometê-la; III - é perseguido logo após; IV - é encontrado logo depois com instrumentos, armas ou objetos que presumam autoria.",
     "fonte": "Decreto-Lei nº 3.689/1941 - CPP", "categoria": "processo_penal"},
    {"id": "doc009", "titulo": "ECA - Art. 112 - Medidas Socioeducativas",
     "texto": "Art. 112. O adolescente infrator pode receber: advertência, reparação do dano, prestação de serviços à comunidade, liberdade assistida, semi-liberdade ou internação.",
     "fonte": "Lei nº 8.069/1990 - ECA", "categoria": "eca"},
    {"id": "doc010", "titulo": "Lei Anticorrupção - Arts. 2º e 3º",
     "texto": "Art. 2º As pessoas jurídicas são responsabilizadas objetivamente pelos atos lesivos em seu benefício. Art. 3º A responsabilização da pessoa jurídica não exclui a responsabilidade individual de seus dirigentes ou administradores.",
     "fonte": "Lei nº 12.846/2013", "categoria": "anticorrupcao"},
]

# Tenta carregar do arquivo JSON; se não encontrar, usa o corpus hardcoded
try:
    with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
        corpus = json.load(f)
    print(f"✅ Corpus carregado do arquivo: {CORPUS_PATH} ({len(corpus)} documentos)")
except FileNotFoundError:
    corpus = CORPUS_HARDCODED
    print(f"⚠️  Arquivo não encontrado. Usando corpus hardcoded ({len(corpus)} documentos).")

print("\n📄 Primeiros 3 documentos:")
for doc in corpus[:3]:
    print(f"  [{doc['id']}] {doc['titulo']}")
    print(f"        {doc['texto'][:80]}...")

---
## PASSO 3 — Criar Índice ChromaDB com Embeddings

Criamos um índice vetorial usando **ChromaDB** e o modelo **paraphrase-multilingual-MiniLM-L12-v2** para gerar embeddings em português.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any

# Carrega o modelo de embeddings multilíngue (384 dimensões)
print("Carregando modelo de embeddings...")
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Modelo de embeddings carregado!")

# Inicializa o ChromaDB em memória
chroma_client = chromadb.Client()

# Cria (ou recria) a coleção com similaridade cosseno
try:
    chroma_client.delete_collection("corpus_juridico")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="corpus_juridico",
    metadata={"hnsw:space": "cosine"}
)

# Gera embeddings em batch para todos os documentos
print("\nGerando embeddings e indexando documentos...")
textos    = [doc['texto']    for doc in corpus]
ids       = [doc['id']       for doc in corpus]
metadados = [{"titulo": doc['titulo'], "fonte": doc['fonte'], "categoria": doc['categoria']} for doc in corpus]

embeddings = embedding_model.encode(textos, show_progress_bar=True)

# Insere no ChromaDB
collection.add(
    documents=textos,
    embeddings=embeddings.tolist(),
    ids=ids,
    metadatas=metadados
)

print(f"\n✅ {len(corpus)} documentos indexados no ChromaDB!")
print(f"   Dimensão dos embeddings: {embeddings.shape[1]}")

---
## PASSO 4 — Conectar ao vLLM e Implementar Detecção de Tokens

Conectamos ao servidor vLLM via `langchain-openai`, apontando `openai_api_base` para `localhost`. Em seguida, implementamos a detecção dos tokens de controle Self-RAG via regex.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import re
from dataclasses import dataclass, field
from typing import Optional

# ─── Cliente vLLM (API OpenAI-compatible) ─────────────────────────────────────
# openai_api_key="EMPTY": o servidor vLLM local não exige chave real
# openai_api_base: aponta para o servidor vLLM na porta 8000
# model_name: deve corresponder exatamente ao modelo carregado no servidor
llm = ChatOpenAI(
    model_name=MODELO,
    openai_api_key="EMPTY",
    openai_api_base=f"http://localhost:{PORTA}/v1",
    temperature=0.1,    # Baixa temperatura = respostas mais determinísticas
    max_tokens=512,
)

# Teste de conectividade
print("Testando conexão com o servidor vLLM...")
resp_teste = llm.invoke("Responda em uma palavra: qual é o idioma oficial do Brasil?")
print(f"✅ Conexão OK! Resposta teste: '{resp_teste.content.strip()}'")  
print(f"   Modelo ativo: {MODELO}")

In [ ]:
# ─── Dataclass para armazenar tokens detectados ───────────────────────────────
@dataclass
class SelfRAGTokens:
    """Estrutura de dados para tokens de controle Self-RAG detectados numa resposta."""
    retrieve: Optional[str] = None       # 'yes' ou 'no'
    isrel:    Optional[str] = None       # 'relevant' ou 'irrelevant'
    issup:    Optional[str] = None       # 'supported', 'partially_supported' ou 'unsupported'
    isuse:    Optional[str] = None       # 'useful' ou 'not_useful'
    texto_resposta: str = ""
    tokens_brutos: Dict[str, str] = field(default_factory=dict)


def detectar_tokens_selfrag(texto: str) -> SelfRAGTokens:
    """
    Detecta tokens de controle Self-RAG num texto de resposta usando regex.
    Aceita variações como [Retrieve: yes], [Retrieve=yes], [Retrieve yes] etc.
    """
    resultado = SelfRAGTokens(texto_resposta=texto)

    # [Retrieve: yes/no]
    m = re.search(r'\[Retrieve[:\s=]+(\w[\w\s]*?)\]', texto, re.IGNORECASE)
    if m:
        v = m.group(1).strip().lower()
        resultado.retrieve = "yes" if any(p in v for p in ["yes", "sim"]) else "no"
        resultado.tokens_brutos["Retrieve"] = m.group(0)

    # [ISREL: relevant/irrelevant]
    m = re.search(r'\[ISREL[:\s=]+([\w\s]+?)\]', texto, re.IGNORECASE)
    if m:
        v = m.group(1).strip().lower()
        resultado.isrel = "relevant" if ("relev" in v and "irr" not in v) else "irrelevant"
        resultado.tokens_brutos["ISREL"] = m.group(0)

    # [ISSUP: supported/partially supported/unsupported]
    m = re.search(r'\[ISSUP[:\s=]+([\w\s]+?)\]', texto, re.IGNORECASE)
    if m:
        v = m.group(1).strip().lower()
        if "partial" in v:
            resultado.issup = "partially_supported"
        elif "unsupport" in v or "not sup" in v:
            resultado.issup = "unsupported"
        else:
            resultado.issup = "supported"
        resultado.tokens_brutos["ISSUP"] = m.group(0)

    # [ISUSE: useful/not useful]
    m = re.search(r'\[ISUSE[:\s=]+([\w\s]+?)\]', texto, re.IGNORECASE)
    if m:
        v = m.group(1).strip().lower()
        resultado.isuse = "not_useful" if "not" in v else "useful"
        resultado.tokens_brutos["ISUSE"] = m.group(0)

    return resultado


def exibir_tokens(tokens: SelfRAGTokens, query: str):
    """Exibe os tokens detectados de forma visual."""
    print(f"\n{'='*60}")
    print(f"📋 QUERY: {query}")
    print('='*60)
    r_icon = "🔍" if tokens.retrieve == "yes" else "⏭️"
    print(f"{r_icon} [Retrieve]: {tokens.retrieve or 'não detectado'}")
    if tokens.retrieve == "yes":
        print(f"{'✅' if tokens.isrel == 'relevant' else '❌'} [ISREL]:    {tokens.isrel or 'não detectado'}")
        print(f"{'✅' if tokens.issup == 'supported' else '⚠️'} [ISSUP]:    {tokens.issup or 'não detectado'}")
        print(f"{'✅' if tokens.isuse == 'useful'    else '❌'} [ISUSE]:    {tokens.isuse or 'não detectado'}")


# Teste rápido da detecção com exemplos simulados
print("🧪 Testando detecção de tokens...")
t1 = detectar_tokens_selfrag("[Retrieve: yes] Preciso dos docs. [ISREL: relevant] Relevante. [ISSUP: supported] Suportado. [ISUSE: useful]")
t2 = detectar_tokens_selfrag("[Retrieve: no] Posso responder com conhecimento geral. [ISSUP: supported] [ISUSE: useful]")
exibir_tokens(t1, "Prazo prescricional (com retrieval)")
exibir_tokens(t2, "O que é princípio da legalidade (sem retrieval)")
print("\n✅ Detecção de tokens funcionando!")

---
## PASSO 5 — Pipeline Self-RAG Completo

Implementamos o pipeline Self-RAG com:
- **Avaliação inicial**: o LLM (via vLLM) decide se precisa buscar documentos
- **Retrieval condicional**: busca no ChromaDB apenas quando necessário
- **Loop de refinamento**: até 3 iterações para melhorar a resposta
- **Critério de parada**: resposta suportada (`[ISSUP=supported]`) e útil (`[ISUSE=useful]`)

In [ ]:
import time

# ─── Prompts do pipeline ──────────────────────────────────────────────────────

PROMPT_AVALIAR_RETRIEVAL = PromptTemplate(
    input_variables=["query"],
    template="""Você é um assistente jurídico Self-RAG.

Analise a pergunta e decida se precisa buscar documentos legais específicos.

Pergunta: {query}

INSTRUÇÕES:
- Se a pergunta pede artigos, prazos, procedimentos ou jurisprudência específicos → emita [Retrieve: yes]
- Se a pergunta é sobre conceitos gerais que você já conhece → emita [Retrieve: no]

Comece sua resposta com [Retrieve: yes] ou [Retrieve: no] e justifique brevemente."""
)

PROMPT_GERAR_COM_CONTEXTO = PromptTemplate(
    input_variables=["query", "contexto", "iteracao"],
    template="""Você é um assistente jurídico Self-RAG (iteração {iteracao}/3).

PERGUNTA: {query}

DOCUMENTOS RECUPERADOS:
{contexto}

INSTRUÇÕES:
1. Avalie a relevância dos documentos: emita [ISREL: relevant] ou [ISREL: irrelevant]
2. Gere uma resposta fundamentada nos documentos em português
3. Avalie o suporte: emita [ISSUP: supported], [ISSUP: partially supported] ou [ISSUP: unsupported]
4. Avalie a utilidade: emita [ISUSE: useful] ou [ISUSE: not useful]

Inclua os tokens de controle na sua resposta."""
)

PROMPT_GERAR_SEM_CONTEXTO = PromptTemplate(
    input_variables=["query"],
    template="""Você é um assistente jurídico Self-RAG.

PERGUNTA: {query}

Você avaliou que pode responder diretamente com conhecimento geral.
Forneça uma resposta clara em português.
Ao final, emita [ISSUP: supported] ou [ISSUP: unsupported] e [ISUSE: useful] ou [ISUSE: not useful]."""
)


# ─── Funções auxiliares ───────────────────────────────────────────────────────

def buscar_documentos(query: str, n: int = 3) -> List[Dict]:
    """Recupera os n documentos mais relevantes do ChromaDB para a query."""
    q_emb = embedding_model.encode([query])[0]
    res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=n,
        include=['documents', 'metadatas', 'distances']
    )
    return [
        {
            'texto': res['documents'][0][i],
            'titulo': res['metadatas'][0][i]['titulo'],
            'similaridade': 1 - res['distances'][0][i]
        }
        for i in range(len(res['documents'][0]))
    ]


def formatar_contexto(docs: List[Dict]) -> str:
    """Formata lista de documentos para inserção no prompt."""
    return "\n\n".join(
        f"[Doc {i+1} | sim={d['similaridade']:.2f}]\n{d['titulo']}\n{d['texto']}"
        for i, d in enumerate(docs)
    )


print("✅ Prompts e funções auxiliares definidos!")

In [ ]:
def pipeline_self_rag(query: str, max_iteracoes: int = 3, verbose: bool = True) -> Dict[str, Any]:
    """
    Executa o pipeline Self-RAG completo para uma query usando o servidor vLLM.

    Etapas:
      1. Avalia necessidade de retrieval via vLLM  [Retrieve]
      2. Se sim → busca ChromaDB → gera com contexto → avalia [ISREL][ISSUP][ISUSE]
      3. Se não → gera diretamente → avalia [ISSUP][ISUSE]
      4. Itera até max_iteracoes ou até qualidade satisfatória
    """
    inicio = time.time()
    resultado = {
        'query': query,
        'retrieval_ativado': False,
        'iteracoes_realizadas': 0,
        'documentos_recuperados': [],
        'tokens_detectados': [],
        'resposta_final': '',
        'tempo_total': 0
    }

    if verbose:
        print(f"\n{'🔄 '*3} SELF-RAG PIPELINE (vLLM) {'🔄 '*3}")
        print(f"Query: {query}\n")

    # ── Etapa 1: Avaliação de Retrieval ────────────────────────────────────────
    if verbose:
        print("[Etapa 1] Avaliando necessidade de retrieval via vLLM...")

    resp_avaliacao = llm.invoke(PROMPT_AVALIAR_RETRIEVAL.format(query=query)).content
    tokens_aval    = detectar_tokens_selfrag(resp_avaliacao)
    retrieve_sim   = tokens_aval.retrieve == "yes"

    resultado['retrieval_ativado'] = retrieve_sim
    resultado['tokens_detectados'].append(tokens_aval)

    if verbose:
        status = "🔍 ATIVADO" if retrieve_sim else "⏭️  DESATIVADO"
        print(f"   [Retrieve] = {status}")
        print(f"   Justificativa: {resp_avaliacao[:120].replace(chr(10), ' ')}...")

    # ── Etapa 2: Loop de Geração ───────────────────────────────────────────────
    resposta_atual = ""

    for it in range(1, max_iteracoes + 1):
        resultado['iteracoes_realizadas'] = it
        if verbose:
            print(f"\n[Etapa 2 — Iteração {it}/{max_iteracoes}] Gerando resposta via vLLM...")

        if retrieve_sim:
            # Recupera documentos do ChromaDB
            docs = buscar_documentos(query, n=3)
            resultado['documentos_recuperados'] = docs
            if verbose:
                for d in docs:
                    print(f"   🗂  {d['titulo']} (sim={d['similaridade']:.2f})")
            contexto = formatar_contexto(docs)
            prompt   = PROMPT_GERAR_COM_CONTEXTO.format(query=query, contexto=contexto, iteracao=it)
        else:
            prompt = PROMPT_GERAR_SEM_CONTEXTO.format(query=query)

        # Chamada ao servidor vLLM
        resposta_atual = llm.invoke(prompt).content
        tokens_ger     = detectar_tokens_selfrag(resposta_atual)
        resultado['tokens_detectados'].append(tokens_ger)

        if verbose:
            print(f"   ISSUP={tokens_ger.issup} | ISUSE={tokens_ger.isuse}")

        # Critério de parada: suporte OK + útil
        if tokens_ger.issup in ["supported", "partially_supported"] and tokens_ger.isuse == "useful":
            if verbose:
                print(f"   ✅ Qualidade satisfatória na iteração {it}.")
            break
        elif it < max_iteracoes and verbose:
            print("   ⚠️  Qualidade insuficiente. Nova iteração...")

    # Remove tokens do texto final para exibição limpa
    resultado['resposta_final'] = re.sub(r'\[[A-Z]+[:\s=]+[\w\s]+\]', '', resposta_atual).strip()
    resultado['tempo_total'] = round(time.time() - inicio, 2)
    return resultado


print("✅ Função pipeline_self_rag() pronta!")

---
## PASSO 6 — Testar com 4 Queries

**Queries que DEVEM ativar o retrieval** (precisam de documentos específicos):
1. Prazo prescricional — requer Art. 206 CC
2. Flagrante delito — requer Art. 302 CPP

**Queries que NÃO DEVEM ativar o retrieval** (conceitos gerais):
3. Conceito geral de ato ilícito
4. Princípio da legalidade

In [ ]:
QUERIES_TESTE = [
    {"query": "Qual é o prazo prescricional para a pretensão de reparação civil no Código Civil?",
     "espera_retrieval": True,  "descricao": "Específica — requer Art. 206 CC"},
    {"query": "Quais situações configuram flagrante delito segundo o Código de Processo Penal?",
     "espera_retrieval": True,  "descricao": "Específica — requer Art. 302 CPP"},
    {"query": "O que é um ato ilícito de forma geral no direito civil?",
     "espera_retrieval": False, "descricao": "Conceito geral"},
    {"query": "Explique o princípio da legalidade no direito penal brasileiro",
     "espera_retrieval": False, "descricao": "Princípio geral"},
]

todos_resultados = []

for i, item in enumerate(QUERIES_TESTE, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}/4 — {item['descricao']}")
    print(f"Retrieval esperado: {'SIM' if item['espera_retrieval'] else 'NÃO'}")
    print('='*70)

    res = pipeline_self_rag(query=item['query'], max_iteracoes=3, verbose=True)
    res['espera_retrieval'] = item['espera_retrieval']
    res['descricao'] = item['descricao']
    todos_resultados.append(res)

    print(f"\n📝 RESPOSTA FINAL:")
    print(res['resposta_final'][:400] + ("..." if len(res['resposta_final']) > 400 else ""))
    print(f"⏱️  Tempo: {res['tempo_total']}s")

print(f"\n\n✅ Todas as {len(QUERIES_TESTE)} queries processadas!")

---
## PASSO 7 — Tabela Comparativa e Análise

In [ ]:
import pandas as pd
from IPython.display import display

dados_tabela = []
for i, res in enumerate(todos_resultados, 1):
    correto = res['retrieval_ativado'] == res['espera_retrieval']
    dados_tabela.append({
        'Query': f"Q{i}: {res['query'][:55]}...",
        'Retrieval Esperado': '✅ SIM'    if res['espera_retrieval']  else '❌ NÃO',
        'Retrieval Real':     '🔍 ATIVADO' if res['retrieval_ativado'] else '⏭️  DESATIVADO',
        'Comportamento':      '✅ CORRETO' if correto                  else '❌ INCORRETO',
        'Iterações':          res['iteracoes_realizadas'],
        'Docs Recuperados':   len(res['documentos_recuperados']),
        'Tempo (s)':          res['tempo_total']
    })

df = pd.DataFrame(dados_tabela)
print("📊 TABELA COMPARATIVA — Self-RAG via vLLM")
print('='*80)
display(df)

# Estatísticas
n_ativado  = sum(1 for r in todos_resultados if r['retrieval_ativado'])
n_corretos = sum(1 for r in todos_resultados if r['retrieval_ativado'] == r['espera_retrieval'])
t_medio    = sum(r['tempo_total'] for r in todos_resultados) / len(todos_resultados)

print(f"\n📈 RESUMO:")
print(f"   Retrieval ativado:       {n_ativado}/{len(todos_resultados)} queries")
print(f"   Comportamento correto:   {n_corretos}/{len(todos_resultados)} ({100*n_corretos//len(todos_resultados)}%)")
print(f"   Tempo médio por query:   {t_medio:.1f}s")
print(f"   Backend de inferência:   vLLM ({MODELO})")

# Gráfico ASCII de distribuição
print(f"\n📊 DISTRIBUIÇÃO DE RETRIEVAL:")
print(f"   Ativado:    {'█' * n_ativado}{'░' * (len(todos_resultados)-n_ativado)} ({n_ativado})")
print(f"   Desativado: {'░' * n_ativado}{'█' * (len(todos_resultados)-n_ativado)} ({len(todos_resultados)-n_ativado})")

---
## PASSO 8 — Exercício Prático

### 🎯 Crie sua própria query e observe os tokens emitidos!

Substitua `MINHA_QUERY` abaixo e observe:
1. O vLLM ativa `[Retrieve: yes]` ou `[Retrieve: no]`?
2. Os documentos recuperados são `[ISREL: relevant]`?
3. A resposta tem `[ISSUP: supported]`?
4. A resposta é `[ISUSE: useful]`?

**Sugestões de queries para experimentar:**
```python
"O que configura improbidade administrativa?"
"Quais são os direitos do preso na Constituição Federal?"
"Como funciona o sistema jurídico brasileiro?"
"Qual a pena para homicídio qualificado?"
"Quais são as medidas socioeducativas do ECA?"
```

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🎯 EXERCÍCIO: Substitua a query abaixo pela sua pergunta!
# ════════════════════════════════════════════════════════════════

MINHA_QUERY = "Quais são as medidas socioeducativas previstas no ECA?"

# ════════════════════════════════════════════════════════════════

print(f"🚀 Executando Self-RAG via vLLM para:")
print(f"   '{MINHA_QUERY}'\n")

meu_resultado = pipeline_self_rag(MINHA_QUERY, max_iteracoes=3, verbose=True)

print(f"\n{'='*60}")
print("📊 ANÁLISE DOS TOKENS DA SUA QUERY")
print('='*60)
print(f"\n🔄 Retrieval ativado? {'SIM 🔍' if meu_resultado['retrieval_ativado'] else 'NÃO ⏭️'}")
print(f"🔢 Iterações: {meu_resultado['iteracoes_realizadas']}")
print(f"⚙️  Backend: vLLM ({MODELO})")

if meu_resultado['documentos_recuperados']:
    print(f"\n📄 Documentos recuperados:")
    for d in meu_resultado['documentos_recuperados']:
        print(f"   [{d['similaridade']:.2f}] {d['titulo']}")

print(f"\n📝 Resposta Final:")
print(meu_resultado['resposta_final'])
print(f"\n⏱️  Tempo total: {meu_resultado['tempo_total']}s")

print(f"\n💡 REFLEXÃO:")
if meu_resultado['retrieval_ativado']:
    print("   O vLLM considerou que sua pergunta requer documentos específicos")
    print("   e ativou o retrieval no ChromaDB.")
else:
    print("   O vLLM considerou que pode responder diretamente com")
    print("   conhecimento geral, sem buscar documentos externos.")
print("\n   Isso demonstra a DECISÃO AUTÔNOMA do Self-RAG:")
print("   o modelo sabe quando precisa de ajuda e quando já tem a resposta!")

---
## Conclusão

Neste laboratório, implementamos um pipeline **Self-RAG completo** com **vLLM como backend de inferência**, aplicado ao domínio jurídico brasileiro.

### O que foi demonstrado:

1. **vLLM como servidor local**: subimos o servidor com API OpenAI-compatible e consumimos via `langchain-openai` sem alterar o código do pipeline
2. **Tokens de controle** (`[Retrieve]`, `[ISREL]`, `[ISSUP]`, `[ISUSE]`) permitem decisões autônomas sobre quando e como usar retrieval
3. **Retrieval seletivo**: perguntas sobre artigos específicos ativam o ChromaDB; conceitos gerais são respondidos direto pelo LLM
4. **Loop de refinamento**: até 3 iterações para atingir qualidade satisfatória

### Vantagem do vLLM sobre Ollama:
O vLLM usa **PagedAttention** para gerenciar a VRAM de forma eficiente e suporta **batching contínuo** de requisições — fundamental para os múltiplos loops de avaliação do Self-RAG em produção.

### Próximos Passos:
- **LAB2**: Implementar CRAG completo com LangGraph e StateGraph
- Trocar `MODELO` por `selfrag/selfrag_llama2_7b` para tokens nativos (requer GPU A100)
- Expandir o corpus com legislação estadual e jurisprudência dos tribunais

---

*MBA em RAG & CAG Aplicados a Direito e Segurança Pública · Aula 8*